In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns
import time
T1 = time.time()

# 指定分号为字段分隔符
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/4-2/03870.csv', sep=';') 
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/40/M2333_gpsdistance_normalized.csv') #10条
fdata = pd.read_csv('E:/大数据/毕业设计/数据/M3353_gpsdistance_normalized(2).csv') #10条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/datas/10/M5583_gpsdistance_normalized.csv') #20条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/4-2/M2333.csv', sep=';')  #40条
#fdata = pd.read_csv('E:/大数据/毕业设计/数据/result/M4123_dropdata.csv')   #10条
display(fdata)

,idx,opath,lineName,direction,t_flag,time,lng,lat,distance_of_gpspoint,gps_normalized_distance_length
0,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:34:41,113.904367,22.716491,0.001102,0.000035
1,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:34:43,113.904453,22.716505,0.010068,0.000321
2,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:34:56,113.904968,22.716651,0.065409,0.002086
3,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:35:09,113.905589,22.716824,0.132006,0.004210
4,BS09298D,83981.0,M3353,2.0,1,2019-01-04 07:35:23,113.906252,22.717010,0.203158,0.006479
...,...,...,...,...,...,...,...,...,...,...
83307,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:15:44,113.838879,22.688601,31.403388,1.001568
83308,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:15:59,113.839108,22.688451,31.374534,1.000648
83309,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:16:05,113.839414,22.688249,31.336012,0.999419
83310,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:16:14,113.839468,22.688214,31.329220,0.999203


In [2]:
import pandas as pd

# 定义 assign_to_windows 函数
def assign_to_windows(row):
    current_time = row['time']
    minutes_since_midnight = (current_time - current_time.normalize()).total_seconds() / 60.0
    
    # 计算基本时间窗口的开始时间，确保每个窗口覆盖4分钟
    window_start_minute = int(minutes_since_midnight // 4 * 4)  # 使用整除并乘以4确保步进为4
    window_end_minute = window_start_minute + 4  # 窗口结束时间调整为开始时间后的4分钟
    
    return pd.Series([window_start_minute, window_end_minute])

# 将 'time' 列的数据类型从字符串转换为 datetime
fdata['time'] = pd.to_datetime(fdata['time'])

# 应用函数，并创建新的列来存储分配结果
fdata[['win_st', 'win_en']] = fdata.apply(assign_to_windows, axis=1)

In [3]:
# 基于车牌号（'idx'列）、线路号（'linenumber'列）和时间窗口的开始时间（'win_st'列）对数据进行分组
counts_per_group = fdata.groupby(['lineName', 'idx', 'win_st']).size()

# 过滤掉那些在同一时间窗口内每组只有一条记录的数据
# 这会返回一个包含每组在每个时间窗口内记录数大于1的过滤后的DataFrame的索引
filtered_idx = counts_per_group[counts_per_group > 1].index

# 基于过滤后的索引，我们选择原始数据集中符合条件的行
# 这里的'linenumber', 'idx'和'win_st'必须与groupby中使用的列名完全匹配
fdata_filtered = fdata[fdata.set_index(['lineName', 'idx', 'win_st']).index.isin(filtered_idx)]

display(fdata_filtered)

,idx,opath,lineName,direction,t_flag,time,lng,lat,distance_of_gpspoint,gps_normalized_distance_length,win_st,win_en
0,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:34:41,113.904367,22.716491,0.001102,0.000035,452,456
1,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:34:43,113.904453,22.716505,0.010068,0.000321,452,456
2,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:34:56,113.904968,22.716651,0.065409,0.002086,452,456
3,BS09298D,45649.0,M3353,2.0,1,2019-01-04 07:35:09,113.905589,22.716824,0.132006,0.004210,452,456
4,BS09298D,83981.0,M3353,2.0,1,2019-01-04 07:35:23,113.906252,22.717010,0.203158,0.006479,452,456
...,...,...,...,...,...,...,...,...,...,...,...,...
83307,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:15:44,113.838879,22.688601,31.403388,1.001568,552,556
83308,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:15:59,113.839108,22.688451,31.374534,1.000648,552,556
83309,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:16:05,113.839414,22.688249,31.336012,0.999419,556,560
83310,BS40536D,25714.0,M3353,2.0,2,2019-01-04 09:16:14,113.839468,22.688214,31.329220,0.999203,556,560


In [5]:
# 辅助函数，计算两点确定的直线的参数（斜率和截距）
def calculate_line_parameters(point1, point2):
    x_coords, y_coords = [point1[0], point2[0]], [point1[1], point2[1]]
    A = np.vstack([x_coords, np.ones(len(x_coords))]).T
    m, c = np.linalg.lstsq(A, y_coords, rcond=None)[0]
    return m, c

# 辅助函数，计算点到直线的距离
def point_to_line_distance(point, line_points):
    x0, y0 = point
    x1, y1 = line_points[0]
    x2, y2 = line_points[1]
    m, c = calculate_line_parameters((x1, y1), (x2, y2))
    A, B, C = -m, 1, -c
    distance = abs(A * x0 + B * y0 + C) / (np.sqrt(A**2 + B**2))
    return distance

# 函数，计算两条线段之间的最大垂直距离
def calculate_max_vertical_distance_between_lines(line1_points, line2_points):
    point1_line2, point2_line2, midpoint_line2 = line2_points
    distance_start = point_to_line_distance(point1_line2, line1_points)
    distance_mid = point_to_line_distance(midpoint_line2, line1_points)
    distance_end = point_to_line_distance(point2_line2, line1_points)
    return max(distance_start, distance_mid, distance_end)

# 检测公交车串车的函数
def detect_bus_bunching(data, threshold):
    bunching_events = []
    
    for (lineName, win_st, direction), window_data in data.groupby(['lineName', 'win_st', 'direction']):
        # 获取每辆车的首尾位置
        lines = window_data.groupby('idx').agg({
            'gps_normalized_distance_length': ['first', 'last'], 
            'time': ['first', 'last']
        }).reset_index()

        lines.columns = ['idx', 'start_dis', 'end_dis', 'start_time', 'end_time']
        
        lines['start_time'] = lines['start_time'].apply(lambda x: x.timestamp())
        lines['end_time'] = lines['end_time'].apply(lambda x: x.timestamp())
        
        # 检查每对车辆之间的串车情况
        for i in range(len(lines) - 1):
            for j in range(i + 1, len(lines)):
                line1_points = [(lines.at[i, 'start_time'], lines.at[i, 'start_dis']), 
                                (lines.at[i, 'end_time'], lines.at[i, 'end_dis'])]
                line2_points = [(lines.at[j, 'start_time'], lines.at[j, 'start_dis']), 
                                (lines.at[j, 'end_time'], lines.at[j, 'end_dis'])]
                midpoint_line2 = ((line2_points[0][0] + line2_points[1][0]) / 2, 
                                  (line2_points[0][1] + line2_points[1][1]) / 2)

                distance = calculate_max_vertical_distance_between_lines(line1_points, [line2_points[0], midpoint_line2, line2_points[1]])
                
                # 计算串车持续时间
                min_time = min(lines.at[i, 'start_time'], lines.at[j, 'start_time'])
                max_time = max(lines.at[i, 'end_time'], lines.at[j, 'end_time'])
                duration = max_time - min_time

                if distance < threshold:
                    bunching_events.append({
                        'lineName': lineName,
                        'direction': direction,
                        'idx1': lines.at[i, 'idx'],
                        'idx2': lines.at[j, 'idx'],
                        'win_st': win_st,
                        'distance': distance,
                        'start_time': datetime.fromtimestamp(min_time),
                        'end_time': datetime.fromtimestamp(max_time),
                        'duration': timedelta(seconds=duration)
                    })

    return pd.DataFrame(bunching_events)

# 函数，计算两条线段之间的最大垂直距离
def calculate_max_vertical_distance_between_lines(line1_points, line2_points):
    def point_to_line_distance(point, line_points):
        x0, y0 = point
        x1, y1 = line_points[0]
        x2, y2 = line_points[1]
        m, c = np.linalg.lstsq(np.vstack([x_coords, np.ones(len(x_coords))]).T, y_coords, rcond=None)[0]
        A, B, C = -m, 1, -c
        return abs(A * x0 + B * y0 + C) / np.sqrt(A**2 + B**2)

    x_coords, y_coords = [line1_points[0][0], line1_points[1][0]], [line1_points[0][1], line1_points[1][1]]
    point1_line2, point2_line2, midpoint_line2 = line2_points
    distance_start = point_to_line_distance(point1_line2, line1_points)
    distance_mid = point_to_line_distance(midpoint_line2, line1_points)
    distance_end = point_to_line_distance(point2_line2, line1_points)
    return max(distance_start, distance_mid, distance_end)

# 检测公交车串车
threshold = 0.01
bunching_df = detect_bus_bunching(fdata_filtered, threshold)

# 合并串车事件
def merge_bunching_events(events):
    events = events.sort_values(by='win_st').reset_index(drop=True)
    merged_events = []
    i = 0

    while i < len(events):
        current_event = events.iloc[i]
        j = i + 1

        while j < len(events):
            next_event = events.iloc[j]
            if (current_event['idx1'] == next_event['idx1'] and current_event['idx2'] == next_event['idx2'] and
                (next_event['start_time'] - current_event['end_time']).total_seconds() < 5):
                current_event['end_time'] = next_event['end_time']
                current_event['duration'] = current_event['end_time'] - current_event['start_time']
                current_event['win_st'] = f"{current_event['win_st']},{next_event['win_st']}"
                j += 1
            else:
                break

        merged_events.append(current_event)
        i = j

    merged_df = pd.DataFrame(merged_events)
    # 过滤掉持续时间小于3分钟的事件
    filtered_merged_df = merged_df[merged_df['duration'] >= timedelta(minutes=3)]
    return filtered_merged_df

# 合并串车事件
merged_bunching_df = merge_bunching_events(bunching_df)

# 输出合并后的串车事件
print(merged_bunching_df)

T2 = time.time()
print('程序运行时间:%s秒' % ((T2 - T1)*1))

    lineName  direction      idx1      idx2  win_st  distance  \
0      M3353        2.0  BS09075D  BS30452D     368  0.002528   
2      M3353        1.0  BS45786D  BS48256D     448  0.007487   
7      M3353        1.0  BS39179D  BS47985D     484  0.006212   
8      M3353        1.0  BS31509D  BS36216D     488  0.009904   
20     M3353        2.0  BS09053D  BS09691D     496  0.007221   
..       ...        ...       ...       ...     ...       ...   
411    M3353        1.0  BS09390D  BS09945D    1308  0.002613   
412    M3353        1.0  BS08968D  BS09945D    1312  0.002415   
414    M3353        1.0  BS09298D  BS09945D    1324  0.006681   
416    M3353        1.0  BS09053D  BS30452D    1328  0.002557   
417    M3353        1.0  BS09298D  BS30452D    1328  0.005696   

             start_time            end_time        duration  
0   2019-01-04 14:08:10 2019-01-04 14:11:48 0 days 00:03:38  
2   2019-01-04 15:28:11 2019-01-04 15:31:56 0 days 00:03:45  
7   2019-01-04 16:04:15 2019-01-0